# Exportación agregada del Data Mart para Streamlit Cloud

Este cuadernillo se conecta a tu base `proyecto_destino`, calcula tablas agregadas pequeñas para el dashboard y las guarda en la carpeta `data/` en formato `.parquet`.

Al terminar, sube a GitHub:

- `streamlit_app.py`
- `requirements.txt`
- la carpeta `data/` generada por este notebook

No subas la base SQL ni el CSV original.

In [1]:
import os
from pathlib import Path

import pandas as pd
import pyodbc

# Carpeta de salida para Streamlit Cloud
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Credenciales tomadas del notebook del Data Mart
SERVER = "LAPTOP-RSS8LMV8"
DATABASE = "proyecto_destino"
UID = "sa"
PWD = "@DDJJrrvv1303"
DRIVER = "ODBC Driver 17 for SQL Server"

conn_str = (
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID={UID};"
    f"PWD={PWD};"
    "TrustServerCertificate=yes;"
)

conn = pyodbc.connect(conn_str, timeout=30)
print("Conexión exitosa a", DATABASE)

Conexión exitosa a proyecto_destino


In [2]:
# Vista base del Data Mart: NO se exporta completa; solo se usa para agregaciones SQL.
BASE_FROM = """
FROM FactAccidente fa
JOIN DimSeverity      ds   ON fa.severityID      = ds.id
JOIN DimFecha         df   ON fa.fechaID         = df.id
JOIN DimLugar         dl   ON fa.lugarID         = dl.id
JOIN DimClima         dc   ON fa.climaID         = dc.id
JOIN DimCivilTwilight dct  ON fa.civilTwilightID = dct.id
JOIN DimCrossing      dcr  ON fa.crossingID      = dcr.id
JOIN DimJunction      dj   ON fa.junctionID      = dj.id
JOIN DimStation       dst2 ON fa.stationID       = dst2.id
JOIN DimStop          dsp  ON fa.stopID          = dsp.id
JOIN DimTraffic       dtr  ON fa.trafficID       = dtr.id
"""

def export_query(nombre_archivo: str, query: str):
    print(f"Exportando {nombre_archivo}...")
    df = pd.read_sql(query, conn)
    path = DATA_DIR / nombre_archivo
    df.to_parquet(path, index=False)
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"  ✓ {len(df):,} filas -> {path} ({size_mb:.2f} MB)")
    return df


In [3]:
# 1) KPIs por año
export_query("kpis_anio.parquet", f"""
SELECT
    df.anio,
    COUNT_BIG(*) AS total_accidentes,
    SUM(CASE WHEN ds.severity >= 3 THEN 1 ELSE 0 END) AS accidentes_graves,
    AVG(CAST(ds.severity AS FLOAT)) AS severidad_promedio,
    AVG(CAST(fa.duracion_minutos AS FLOAT)) AS duracion_promedio_min,
    AVG(CAST(fa.distance AS FLOAT)) AS distancia_promedio,
    AVG(CAST(fa.temperature AS FLOAT)) AS temperatura_promedio,
    AVG(CAST(fa.humidity AS FLOAT)) AS humedad_promedio,
    AVG(CAST(fa.visibility AS FLOAT)) AS visibilidad_promedio,
    AVG(CAST(fa.wind_speed AS FLOAT)) AS viento_promedio,
    AVG(CAST(fa.precipitation AS FLOAT)) AS precipitacion_promedio
{BASE_FROM}
GROUP BY df.anio
ORDER BY df.anio
""")

# 2) Tendencia mensual
export_query("accidentes_mes.parquet", f"""
SELECT
    df.anio,
    df.mes,
    COUNT_BIG(*) AS total_accidentes,
    SUM(CASE WHEN ds.severity >= 3 THEN 1 ELSE 0 END) AS accidentes_graves,
    AVG(CAST(ds.severity AS FLOAT)) AS severidad_promedio,
    AVG(CAST(fa.duracion_minutos AS FLOAT)) AS duracion_promedio_min
{BASE_FROM}
GROUP BY df.anio, df.mes
ORDER BY df.anio, df.mes
""")

# 3) Distribución por severidad
export_query("severidad_anio.parquet", f"""
SELECT
    df.anio,
    ds.severity,
    COUNT_BIG(*) AS total_accidentes,
    AVG(CAST(fa.duracion_minutos AS FLOAT)) AS duracion_promedio_min
{BASE_FROM}
GROUP BY df.anio, ds.severity
ORDER BY df.anio, ds.severity
""")

# 4) Heatmap hora x severidad
export_query("hora_severidad.parquet", f"""
SELECT
    df.anio,
    df.hora,
    ds.severity,
    COUNT_BIG(*) AS total_accidentes
{BASE_FROM}
GROUP BY df.anio, df.hora, ds.severity
ORDER BY df.anio, df.hora, ds.severity
""")

# 5) Día de semana vs luz
export_query("dia_luz.parquet", f"""
SELECT
    df.anio,
    df.dia_semana,
    dct.civil_twilight,
    COUNT_BIG(*) AS total_accidentes,
    SUM(CASE WHEN ds.severity >= 3 THEN 1 ELSE 0 END) AS accidentes_graves
{BASE_FROM}
GROUP BY df.anio, df.dia_semana, dct.civil_twilight
ORDER BY df.anio, df.dia_semana, dct.civil_twilight
""")


Exportando kpis_anio.parquet...


C:\Users\Diego\AppData\Local\Temp\ipykernel_25956\101740694.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


  ✓ 8 filas -> data\kpis_anio.parquet (0.01 MB)
Exportando accidentes_mes.parquet...
  ✓ 87 filas -> data\accidentes_mes.parquet (0.01 MB)
Exportando severidad_anio.parquet...
  ✓ 30 filas -> data\severidad_anio.parquet (0.00 MB)
Exportando hora_severidad.parquet...
  ✓ 703 filas -> data\hora_severidad.parquet (0.01 MB)
Exportando dia_luz.parquet...
  ✓ 112 filas -> data\dia_luz.parquet (0.00 MB)


,anio,dia_semana,civil_twilight,total_accidentes,accidentes_graves
0,2016,Friday,Day,53930,16380
1,2016,Friday,Night,16417,6852
2,2016,Monday,Day,52348,16159
3,2016,Monday,Night,15044,5993
4,2016,Saturday,Day,14156,6765
...,...,...,...,...,...
107,2023,Thursday,Night,11657,412
108,2023,Tuesday,Day,27859,682
109,2023,Tuesday,Night,12218,425
110,2023,Wednesday,Day,26041,654


In [4]:
# 6) Estados
export_query("top_estados.parquet", f"""
SELECT
    df.anio,
    dl.state,
    COUNT_BIG(*) AS total_accidentes,
    SUM(CASE WHEN ds.severity >= 3 THEN 1 ELSE 0 END) AS accidentes_graves,
    AVG(CAST(ds.severity AS FLOAT)) AS severidad_promedio,
    AVG(CAST(fa.duracion_minutos AS FLOAT)) AS duracion_promedio_min,
    AVG(CAST(fa.distance AS FLOAT)) AS distancia_promedio
{BASE_FROM}
WHERE dl.state IS NOT NULL
GROUP BY df.anio, dl.state
ORDER BY df.anio, total_accidentes DESC
""")

# 7) Ciudades
export_query("top_ciudades.parquet", f"""
SELECT
    df.anio,
    dl.state,
    dl.city,
    COUNT_BIG(*) AS total_accidentes,
    SUM(CASE WHEN ds.severity >= 3 THEN 1 ELSE 0 END) AS accidentes_graves,
    AVG(CAST(ds.severity AS FLOAT)) AS severidad_promedio,
    AVG(CAST(fa.duracion_minutos AS FLOAT)) AS duracion_promedio_min
{BASE_FROM}
WHERE dl.city IS NOT NULL
GROUP BY df.anio, dl.state, dl.city
HAVING COUNT_BIG(*) >= 5
ORDER BY df.anio, accidentes_graves DESC
""")

# 8) Clima
export_query("clima.parquet", f"""
SELECT
    df.anio,
    dc.weather_condition,
    COUNT_BIG(*) AS total_accidentes,
    SUM(CASE WHEN ds.severity >= 3 THEN 1 ELSE 0 END) AS accidentes_graves,
    AVG(CAST(ds.severity AS FLOAT)) AS severidad_promedio,
    AVG(CAST(fa.duracion_minutos AS FLOAT)) AS duracion_promedio_min,
    AVG(CAST(fa.temperature AS FLOAT)) AS temperatura_promedio,
    AVG(CAST(fa.humidity AS FLOAT)) AS humedad_promedio,
    AVG(CAST(fa.visibility AS FLOAT)) AS visibilidad_promedio,
    AVG(CAST(fa.wind_speed AS FLOAT)) AS viento_promedio,
    AVG(CAST(fa.precipitation AS FLOAT)) AS precipitacion_promedio
{BASE_FROM}
WHERE dc.weather_condition IS NOT NULL
GROUP BY df.anio, dc.weather_condition
HAVING COUNT_BIG(*) >= 5
ORDER BY df.anio, total_accidentes DESC
""")


Exportando top_estados.parquet...


C:\Users\Diego\AppData\Local\Temp\ipykernel_25956\101740694.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


  ✓ 388 filas -> data\top_estados.parquet (0.02 MB)
Exportando top_ciudades.parquet...
  ✓ 56,827 filas -> data\top_ciudades.parquet (0.79 MB)
Exportando clima.parquet...
  ✓ 522 filas -> data\clima.parquet (0.04 MB)


,anio,weather_condition,total_accidentes,accidentes_graves,severidad_promedio,duracion_promedio_min,temperatura_promedio,humedad_promedio,visibilidad_promedio,viento_promedio,precipitacion_promedio
0,2016,Clear,174341,58577,2.369288,167.423784,68.379831,56.612767,9.873337,7.941252,0.249357
1,2016,Overcast,57413,19225,2.374549,276.990281,58.274948,78.033091,8.810604,8.513289,0.066249
2,2016,Mostly Cloudy,53640,18353,2.373695,249.851585,69.240944,65.338844,9.828353,9.113026,0.052964
3,2016,Partly Cloudy,41885,14147,2.361394,209.903474,70.916720,59.119644,9.941757,9.048449,0.037944
4,2016,Scattered Clouds,40287,13957,2.380197,250.545536,73.396694,61.022877,9.992020,9.080016,0.066866
...,...,...,...,...,...,...,...,...,...,...,...
517,2023,Freezing Drizzle,7,0,2.000000,109.000000,32.000000,100.000000,3.000000,0.000000,0.000000
518,2023,Heavy Sleet,6,0,2.000000,138.750000,28.166667,87.500000,1.000000,17.833333,0.038333
519,2023,Blowing Dust,6,0,2.000000,150.000000,55.166667,23.500000,6.500000,14.666667,0.000000
520,2023,Hail,5,0,2.000000,112.750000,46.400000,79.000000,3.600000,17.600000,0.218000


In [5]:
# 9) Infraestructura vial agregada
infra_query = f"""
SELECT anio, infraestructura,
       SUM(total_accidentes) AS total_accidentes,
       SUM(accidentes_graves) AS accidentes_graves,
       CAST(SUM(accidentes_graves) AS FLOAT) / NULLIF(SUM(total_accidentes), 0) * 100 AS tasa_graves_pct
FROM (
    SELECT df.anio, 'Cruce peatonal' AS infraestructura,
           COUNT_BIG(*) AS total_accidentes,
           SUM(CASE WHEN ds.severity >= 3 THEN 1 ELSE 0 END) AS accidentes_graves
    {BASE_FROM}
    WHERE dcr.crossing = 'True'
    GROUP BY df.anio

    UNION ALL
    SELECT df.anio, 'Intersección' AS infraestructura,
           COUNT_BIG(*) AS total_accidentes,
           SUM(CASE WHEN ds.severity >= 3 THEN 1 ELSE 0 END) AS accidentes_graves
    {BASE_FROM}
    WHERE dj.junction = 'True'
    GROUP BY df.anio

    UNION ALL
    SELECT df.anio, 'Estación' AS infraestructura,
           COUNT_BIG(*) AS total_accidentes,
           SUM(CASE WHEN ds.severity >= 3 THEN 1 ELSE 0 END) AS accidentes_graves
    {BASE_FROM}
    WHERE dst2.station = 'True'
    GROUP BY df.anio

    UNION ALL
    SELECT df.anio, 'Señal Stop' AS infraestructura,
           COUNT_BIG(*) AS total_accidentes,
           SUM(CASE WHEN ds.severity >= 3 THEN 1 ELSE 0 END) AS accidentes_graves
    {BASE_FROM}
    WHERE dsp.stop = 'True'
    GROUP BY df.anio

    UNION ALL
    SELECT df.anio, 'Semáforo' AS infraestructura,
           COUNT_BIG(*) AS total_accidentes,
           SUM(CASE WHEN ds.severity >= 3 THEN 1 ELSE 0 END) AS accidentes_graves
    {BASE_FROM}
    WHERE dtr.traffic_signal = 'True'
    GROUP BY df.anio
) x
GROUP BY anio, infraestructura
ORDER BY anio, total_accidentes DESC
"""
export_query("infraestructura.parquet", infra_query)

# 10) Duración por severidad
export_query("duracion_severidad.parquet", f"""
SELECT
    df.anio,
    ds.severity,
    COUNT_BIG(*) AS total_accidentes,
    AVG(CAST(fa.duracion_minutos AS FLOAT)) AS duracion_promedio_min
{BASE_FROM}
GROUP BY df.anio, ds.severity
ORDER BY df.anio, ds.severity
""")

# 11) Duración por estado
export_query("duracion_estado.parquet", f"""
SELECT
    df.anio,
    dl.state,
    COUNT_BIG(*) AS total_accidentes,
    AVG(CAST(fa.duracion_minutos AS FLOAT)) AS duracion_promedio_min
{BASE_FROM}
WHERE dl.state IS NOT NULL
GROUP BY df.anio, dl.state
HAVING COUNT_BIG(*) >= 10
ORDER BY df.anio, duracion_promedio_min DESC
""")

# 12) Duración por clima
export_query("duracion_clima.parquet", f"""
SELECT
    df.anio,
    dc.weather_condition,
    COUNT_BIG(*) AS total_accidentes,
    AVG(CAST(fa.duracion_minutos AS FLOAT)) AS duracion_promedio_min
{BASE_FROM}
WHERE dc.weather_condition IS NOT NULL
GROUP BY df.anio, dc.weather_condition
HAVING COUNT_BIG(*) >= 10
ORDER BY df.anio, duracion_promedio_min DESC
""")


Exportando infraestructura.parquet...


C:\Users\Diego\AppData\Local\Temp\ipykernel_25956\101740694.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


  ✓ 40 filas -> data\infraestructura.parquet (0.00 MB)
Exportando duracion_severidad.parquet...
  ✓ 30 filas -> data\duracion_severidad.parquet (0.00 MB)
Exportando duracion_estado.parquet...
  ✓ 383 filas -> data\duracion_estado.parquet (0.01 MB)
Exportando duracion_clima.parquet...
  ✓ 458 filas -> data\duracion_clima.parquet (0.01 MB)


,anio,weather_condition,total_accidentes,duracion_promedio_min
0,2016,Fair,2361,1951.756459
1,2016,Light Snow,2391,305.679214
2,2016,Overcast,57413,276.990281
3,2016,Snow,232,251.237069
4,2016,Scattered Clouds,40287,250.545536
...,...,...,...,...
453,2023,Heavy Drizzle,12,101.083333
454,2023,Patches of Fog,67,93.452381
455,2023,Freezing Rain,17,92.272727
456,2023,Blowing Snow / Windy,12,74.750000


In [6]:
# Validación final: listar archivos generados y tamaños
print("\nArchivos generados:")
for p in sorted(DATA_DIR.glob("*.parquet")):
    print(f"- {p.as_posix()} | {p.stat().st_size / (1024*1024):.2f} MB")

conn.close()
print("\nListo. Ahora sube a GitHub: streamlit_app.py, requirements.txt y la carpeta data/.")


Archivos generados:
- data/accidentes_mes.parquet | 0.01 MB
- data/clima.parquet | 0.04 MB
- data/dia_luz.parquet | 0.00 MB
- data/duracion_clima.parquet | 0.01 MB
- data/duracion_estado.parquet | 0.01 MB
- data/duracion_severidad.parquet | 0.00 MB
- data/hora_severidad.parquet | 0.01 MB
- data/infraestructura.parquet | 0.00 MB
- data/kpis_anio.parquet | 0.01 MB
- data/severidad_anio.parquet | 0.00 MB
- data/top_ciudades.parquet | 0.79 MB
- data/top_estados.parquet | 0.02 MB

Listo. Ahora sube a GitHub: streamlit_app.py, requirements.txt y la carpeta data/.
